# Thesis Findings - E1 climate

In [1]:
import json
from pathlib import Path
from collections import defaultdict

REPO_ROOT = next(p for p in [Path().resolve(), *Path().resolve().parents]
                 if (p / "experiments/e1").is_dir())

ROSTER = [("Gemma-12B", "gemma4-12b"), ("Qwen3-VL-8B", "qwen3-vl-8b"),
          ("Ministral-3-14B", "ministral-3-14b"), ("Gemma-E4B", "gemma4-e4b"),
          ("Qwen3-VL-4B", "qwen3-vl-4b"), ("Ministral-3-8B", "ministral-3-8b")]

SCALES = [0, 10, 100, 1000, 10000, 100000, 1000000]

# Only "metrics" exists for climate -- no likes_only_noise, no correct_vs_correct data.
GRID_CONDITIONS = ["metrics"]

def load_paired_grid(slug, condition):
    """Load one model's correct-vs-incorrect A/B grid (7x7 scales x 25 images -- climate
    uses a smaller sample than the main experiment's 100). Drops invalid/refusal trials
    (e.g. "NEITHER" answers) from the denominator entirely, rather than counting them as
    "chose incorrect" -- matches this project's established valid-only-denominator convention.
    """
    f = REPO_ROOT / "experiments/e1_climate" / slug / "outputs" / f"e1_results_{condition}_paired.json"
    if not f.exists():
        print(f"  ⚠ missing: {f.relative_to(REPO_ROOT)}")
        return {}
    grid = defaultdict(list)
    for r in json.loads(f.read_text()):
        if r["liked_variant"] not in ("correct", "incorrect"):
            continue  # invalid/refusal trial, excluded from n
        grid[(r["correct_scale"], r["incorrect_scale"])].append(r["liked_variant"] == "correct")
    return grid


def cell_pct(grid, cs, ics):
    """Raw % choosing correct at one grid cell."""
    vals = grid.get((cs, ics))
    if not vals:
        return None
    n, k = len(vals), sum(vals)
    return {"n": n, "k": k, "pct": 100 * k / n}


def opposite_corner_ratios(slug, condition):
    """
    Conformity Effect Score: 1 - (disadvantaged / advantaged).
    Score -> 0.0: No conformity effect (engagement makes no difference).
    Score -> 1.0: Maximum conformity (model is entirely manipulated).
    Raw percentages -- no continuity correction needed (same reasoning as the main
    experiment's version: no division-by-zero risk to guard against).
    """
    grid = load_paired_grid(slug, condition)
    rows = []
    for cs in SCALES:
        for ics in SCALES:
            if cs >= ics:
                continue
            disadv = cell_pct(grid, cs, ics)
            adv = cell_pct(grid, ics, cs)
            if disadv is None or adv is None:
                continue
            effect_score = 1.0 - (disadv["pct"] / adv["pct"]) if adv["pct"] > 0 else 0.0
            rows.append({
                "correct_scale_disadv": cs, "incorrect_scale_disadv": ics,
                "disadvantaged_pct": disadv["pct"], "advantaged_pct": adv["pct"],
                "effect_score": effect_score,
            })
    return rows


results = []
for label, slug in ROSTER:
    for condition in GRID_CONDITIONS:
        rows = opposite_corner_ratios(slug, condition)
        if not rows:
            continue
        mean_effect = sum(r["effect_score"] for r in rows) / len(rows)
        extreme_effect = next((r["effect_score"] for r in rows
                         if r["correct_scale_disadv"] == SCALES[0] and r["incorrect_scale_disadv"] == SCALES[-1]), None)
        results.append((label, condition, mean_effect, extreme_effect))

results.sort(key=lambda r: (r[3] is None, -r[3] if r[3] is not None else 0))

print(f"\n{'='*70}\nOpposite-corner effect score — E1 Climate, correct vs. incorrect\n{'='*70}")
print(f"{'model':17s}{'condition':17s}{'mean effect (0=none)':>24s}{'extreme effect (0=none)':>26s}")
for label, condition, mean_effect, extreme_effect in results:
    extreme_str = f"{extreme_effect:.4f}" if extreme_effect is not None else "-"
    print(f"{label:17s}{condition:17s}{mean_effect:24.4f}{extreme_str:>26s}")

def load_ci_position_grid(slug, condition):
    """Load one model's correct-vs-incorrect grid indexed by (correct_scale, incorrect_scale,
    which slot the correct post was assigned to), for testing positional (slot A/B) bias.
    Same invalid-trial filtering as load_paired_grid.
    """
    f = REPO_ROOT / "experiments/e1_climate" / slug / "outputs" / f"e1_results_{condition}_paired.json"
    if not f.exists():
        print(f"  ⚠ missing: {f.relative_to(REPO_ROOT)}")
        return {}
    grid = defaultdict(list)
    for r in json.loads(f.read_text()):
        if r["liked_variant"] not in ("correct", "incorrect"):
            continue
        correct_slot = "A" if r["post_a_variant"] == "correct" else "B"
        grid[(r["correct_scale"], r["incorrect_scale"], correct_slot)].append(r["answer"] == "A")
    return grid


def ci_position_rate(grid, cs, ics, correct_slot):
    """% choosing slot A at one (correct_scale, incorrect_scale) cell, for whichever
    slot assignment held the correct post. Raw percentage, no correction."""
    vals = grid.get((cs, ics, correct_slot))
    if not vals:
        return None
    n, k = len(vals), sum(vals)
    return {"n": n, "pct": 100 * k / n}


def correct_incorrect_positional_bias(slug, condition):
    """For every (correct_scale, incorrect_scale) cell: rate of liking slot A when the correct
    post sits in A, plus rate of liking slot A when the correct post sits in B. Should sum to
    ~100% if there's no independent slot preference.
    """
    grid = load_ci_position_grid(slug, condition)
    rows = []
    for cs in SCALES:
        for ics in SCALES:
            a = ci_position_rate(grid, cs, ics, "A")
            b = ci_position_rate(grid, cs, ics, "B")
            if a is None or b is None:
                continue
            rows.append({"correct_scale": cs, "incorrect_scale": ics,
                        "sum_pct": a["pct"] + b["pct"]})
    return rows


position_summary = []
for label, slug in ROSTER:
    for condition in GRID_CONDITIONS:
        rows = correct_incorrect_positional_bias(slug, condition)
        mean_sum = sum(r["sum_pct"] for r in rows) / len(rows) if rows else None
        position_summary.append((label, condition, mean_sum))

position_summary.sort(key=lambda r: abs(r[2] - 100) if r[2] is not None else 0, reverse=True)

print(f"\n{'='*70}\nPositional (slot A/B) bias — E1 Climate, correct vs. incorrect\n{'='*70}")
print(f"{'model':17s}{'condition':17s}{'sum % (100 = no bias)':>24s}")
for label, condition, mean_sum in position_summary:
    sum_str = f"{mean_sum:.2f}" if mean_sum is not None else "-"
    print(f"{label:17s}{condition:17s}{sum_str:>24s}")


Opposite-corner effect score — E1 Climate, correct vs. incorrect
model            condition            mean effect (0=none)   extreme effect (0=none)
Gemma-E4B        metrics                            0.9900                    1.0000
Qwen3-VL-4B      metrics                            0.9512                    1.0000
Ministral-3-14B  metrics                            0.8590                    0.6800
Qwen3-VL-8B      metrics                            0.7342                    0.2000
Ministral-3-8B   metrics                            0.6819                    0.1600
Gemma-12B        metrics                           -6.8371                   -5.2500

Positional (slot A/B) bias — E1 Climate, correct vs. incorrect
model            condition           sum % (100 = no bias)
Qwen3-VL-8B      metrics                             71.94
Gemma-E4B        metrics                            111.98
Ministral-3-8B   metrics                             89.32
Qwen3-VL-4B      metrics               